## 完全混合集団ゲーム と 格子上ゲームでの進化動学 比較

戦略：
- C: 協力
- D: 非協力

ゲームの利得行列：
|       | C      | D      |
|-------|--------|--------|
| C     | (5, 5) | (0, 6) |
| D     | (6, 0) | (1, 1) |

モデル：
- 完全混合集団モデル：集団全体の戦略確率によって期待利得が決まる。
- 格子上ゲーム：個体が格子上に配置され、近隣の個体とゲームを行う。戦略は近隣の戦略に影響される。

In [16]:
import random

In [ ]:
class Agent:
    """
    エージェントクラス：

    """
    def __init__(self, strgSet: list, payMatrix: list):
        # 戦略の集合
        self.strgSet = strgSet
        # 初期戦略をランダムに選択
        self.strategy = random.choice(self.strgSet)
        # 次の戦略への遷移確率
        self.switchingRate = None
        # 利得行列
        self.payMatrix = payMatrix
        # 期待利得
        self.expPayOff = {strg: 0 for strg in self.strgSet}

    # 期待利得の計算
    def calc_expPayOff(self, opp_agents: list):
        # opp_agent の戦略の割合を計算
        strg_rate = {strg: 0 for strg in self.strgSet}
        for agent in opp_agents:
            strg_rate[agent.strategy] += 1 / len(opp_agents)
        
        # 期待利得を計算
        for strg in self.strgSet: # 自分の戦略
            for opp_strg in self.strgSet: # 相手の戦略
                self.expPayOff[strg] += self.payMatrix[strg][opp_strg] * strg_rate[opp_strg]

    def IoS(self, opp_agents: list, expPayAdd: float=0.0):
        """
        計算式

        戦略 i から戦略 j に変更する確率：
        P(i → j) = 戦略 j を持つ人の割合 * (戦略 j の期待利得 + 補正値) / (利得行列の最大値 + 補正値)
        """
        # 利得行列の最大値
        maxPay = max(
            value
            for row in self.payMatrix.values()
            for value in row.values()
        )

        # スイッチングレートの初期化
        self.switchingRate = {strg: 0 for strg in self.strgSet}

        # スイッチングレートのリスト
        switching_rates = []
        
        # 現在とは異なる戦略の中から、次の戦略への遷移確立を計算
        for i, to_strg in enumerate([strg for strg in self.strgSet if strg != self.strategy]):
            # 戦略 to_strg を持つ人の割合
            strg_rate = sum(1 for agent in opp_agents if agent.strategy == to_strg) / len(opp_agents)
            
            # 遷移確率を計算
            switching_rates.append(strg_rate * (self.expPayOff[to_strg] + expPayAdd) / (maxPay + expPayAdd))

            self.switchingRate[to_strg] = switching_rates[-1]
        
        # そのまま残る確率を計算
        self.switchingRate[self.strategy] = 1 - sum(switching_rates)


            
    # 戦略の更新
    def update_strategy(self):
        self.strategy = self.next_strategy

In [18]:
# テスト (calc_expPayOff)
strgSet = ["C", "D"]

payMatrix = {
    "C": {"C": 5, "D": 0},
    "D": {"C": 6, "D": 1}
}

agent_list = [Agent(strgSet, payMatrix) for _ in range(10)]
for agent in agent_list:
    agent.calc_expPayOff(agent_list)

expPayOff_list = [agent.expPayOff for agent in agent_list]
print(expPayOff_list)

[{'C': 3.0, 'D': 3.9999999999999996}, {'C': 3.0, 'D': 3.9999999999999996}, {'C': 3.0, 'D': 3.9999999999999996}, {'C': 3.0, 'D': 3.9999999999999996}, {'C': 3.0, 'D': 3.9999999999999996}, {'C': 3.0, 'D': 3.9999999999999996}, {'C': 3.0, 'D': 3.9999999999999996}, {'C': 3.0, 'D': 3.9999999999999996}, {'C': 3.0, 'D': 3.9999999999999996}, {'C': 3.0, 'D': 3.9999999999999996}]


In [ ]:
class Game:
    def __init__(self, strgSet: list, payMatrix: list, L: int=50):
        self.L = L
        self.agent_list = [[Agent(strgSet, payMatrix) for _ in range(L)] for _ in range(L)]

In [ ]:
# class FindNextStrg:
#     """
#     次の戦略を決定するクラス：

#     """
#     def __init__(self, agent_list: list):
#         self.agent_list = agent_list
#         self.L = len(agent_list) # 格子のサイズ

   
        
# # 格子モデル
# def lattice(self, n_neighbors: int=4):
#     for i, row in enumerate(self.agent_list):
#         for j, agent in enumerate(row):

#             neighbors = [
#                 self.agent_list[i-1][j],   # 上
#                 self.agent_list[i+1-self.L][j],   # 下
#                 self.agent_list[i][j-1],   # 左
#                 self.agent_list[i][j+1-self.L]    # 右
#             ]
#             agent.calc_expPayOff(neighbors)

# # 完全混合集合モデル
# def complete_mix(self):
#     opp_agents = [agent for row in self.agent_list for agent in row]
#     for row in self.agent_list:
        # for agent in row:
        #     agent.calc_expPayOff(opp_agents)